# Emotion Recognition System — Data Preprocessing

A reproducible baseline for the YOLO emotion dataset. Source data is read-only; processed data is written to `dataset/processed/`.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import math
import shutil

import cv2
import numpy as np
import pandas as pd
import yaml
from PIL import Image

## 1. Load and validate source inputs

The audit established that the source is already 96×96 RGB, so the baseline keeps that size rather than inventing detail through upscaling.

In [ ]:
SOURCE_ROOT = Path('../dataset/YOLO_format').resolve()
if not SOURCE_ROOT.exists():
    SOURCE_ROOT = Path('dataset/YOLO_format').resolve()
OUTPUT_ROOT = SOURCE_ROOT.parent / 'processed'
TARGET_SIZE = (96, 96)
SPLIT_NAMES = ('train', 'valid', 'test')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

with (SOURCE_ROOT / 'data.yaml').open(encoding='utf-8') as file:
    source_config = yaml.safe_load(file)
class_names = source_config['names']
if isinstance(class_names, dict):
    class_names = [class_names[index] for index in sorted(class_names)]
source_splits = {name: {'images': SOURCE_ROOT / name / 'images', 'labels': SOURCE_ROOT / name / 'labels'} for name in SPLIT_NAMES}
print('Source:', SOURCE_ROOT)
print('Output:', OUTPUT_ROOT)
print('Target:', TARGET_SIZE)
print('Classes:', dict(enumerate(class_names)))

In [ ]:
def source_images(folder):
    return sorted(path for path in folder.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

def validate_yolo_label(path, class_count):
    if not path.exists():
        return False, 'missing label'
    lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not lines:
        return False, 'empty label'
    for line in lines:
        try:
            values = [float(value) for value in line.split()]
            class_id = int(values[0])
            valid = (len(values) == 5 and values[0].is_integer() and 0 <= class_id < class_count and all(math.isfinite(value) for value in values) and all(0 <= value <= 1 for value in values[1:]) and values[3] > 0 and values[4] > 0)
            if not valid:
                return False, 'invalid YOLO values'
        except (ValueError, IndexError):
            return False, 'invalid YOLO row'
    return True, ''

records, input_errors = [], []
for split, folders in source_splits.items():
    for image_path in source_images(folders['images']):
        label_path = folders['labels'] / f'{image_path.stem}.txt'
        try:
            with Image.open(image_path) as image:
                image.load()
                width, height, mode = image.width, image.height, image.mode
            label_valid, issue = validate_yolo_label(label_path, len(class_names))
            records.append({'split': split, 'source_image': image_path, 'source_label': label_path, 'filename': image_path.name, 'stem': image_path.stem, 'width': width, 'height': height, 'mode': mode, 'label_valid': label_valid, 'issue': issue})
        except Exception as error:
            input_errors.append({'split': split, 'filename': image_path.name, 'issue': f'unreadable image: {error}'})
input_table = pd.DataFrame(records)
assert not input_errors and input_table['label_valid'].all(), 'Resolve invalid inputs before processing.'
print(input_table.groupby('split').size())

## 2. Flag duplicate and low-sharpness images

Flags are written to a manifest; no source images are deleted automatically.

In [ ]:
def source_hash(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def sharpness_score(path):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return None if gray is None else float(cv2.Laplacian(gray, cv2.CV_64F).var())

input_table['md5'] = input_table['source_image'].map(source_hash)
input_table['sharpness'] = input_table['source_image'].map(sharpness_score)
input_table['duplicate_group_size'] = input_table.groupby('md5')['md5'].transform('size')
input_table['duplicate_splits'] = input_table.groupby('md5')['split'].transform(lambda values: ','.join(sorted(set(values))))
input_table['cross_split_duplicate'] = input_table['duplicate_splits'].str.contains(',')
input_table['within_split_duplicate'] = input_table.groupby(['split', 'md5'])['md5'].transform('size') > 1
input_table['sharpness_p10'] = input_table.groupby('split')['sharpness'].transform(lambda values: values.quantile(0.10))
input_table['low_sharpness_review'] = input_table['sharpness'] < input_table['sharpness_p10']
display(input_table.groupby('split').agg(low_sharpness_review=('low_sharpness_review', 'sum'), within_split_duplicates=('within_split_duplicate', 'sum'), cross_split_duplicates=('cross_split_duplicate', 'sum')))